In [12]:
import sys
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

IN_COLAB = "google.colab" in sys.modules

DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("model_checkpoint")
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

In [3]:
checkpoint_files = ["tfidf_vectorizer.joblib", "logreg_classifier.joblib", "config.json"]
missing = [f for f in checkpoint_files if not (CHECKPOINT_DIR / f).exists()]

if IN_COLAB and missing:
    from google.colab import files
    print(f"Upload your Stage 1 checkpoint files: {missing}")
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).replace(CHECKPOINT_DIR / fname)

with open(CHECKPOINT_DIR / "config.json") as f:
    stage1_config = json.load(f)

print("Stage 1 checkpoint config:", stage1_config)
assert stage1_config["winning_model"] == "logistic_regression"

vectorizer = joblib.load(CHECKPOINT_DIR / "tfidf_vectorizer.joblib")
clf = joblib.load(CHECKPOINT_DIR / "logreg_classifier.joblib")

print("Loaded vectorizer vocabulary size:", len(vectorizer.vocabulary_))
print("Loaded classifier:", clf)

Upload your Stage 1 checkpoint files: ['tfidf_vectorizer.joblib', 'logreg_classifier.joblib', 'config.json']


Saving config.json to config.json
Saving logreg_classifier.joblib to logreg_classifier.joblib
Saving tfidf_vectorizer.joblib to tfidf_vectorizer.joblib
Stage 1 checkpoint config: {'winning_model': 'logistic_regression', 'public_test_accuracy': 0.715, 'seed': 42, 'decision_threshold': 0.5}
Loaded vectorizer vocabulary size: 6000
Loaded classifier: LogisticRegression(C=0.0005, class_weight='balanced', max_iter=3000,
                   random_state=42)


In [6]:
hidden_test_path = DATA_DIR / "hidden_test.csv"

if IN_COLAB and not hidden_test_path.exists():
    from google.colab import files
    print("Upload hidden_test.csv:")
    uploaded = files.upload()
    for fname in uploaded:
        Path(fname).replace(DATA_DIR / fname)

hidden_test_df = pd.read_csv(hidden_test_path)
print("hidden_test shape:", hidden_test_df.shape)
print(hidden_test_df["label"].value_counts())
hidden_test_df.head()


Upload hidden_test.csv:


Saving hidden_test.csv to hidden_test.csv
hidden_test shape: (600, 5)
label
0    300
1    300
Name: count, dtype: int64


,id,text,label,label_name,source_file
0,neg_cv795_10291,"mr . bean , a bumbling security guard from eng...",0,negative,neg/cv795_10291.txt
1,neg_cv174_9735,"starship troopers is a bad movie . \ni mean , ...",0,negative,neg/cv174_9735.txt
2,pos_cv065_15248,"what a great film . \nwhat a stunning , touchi...",1,positive,pos/cv065_15248.txt
3,neg_cv076_26009,"susan granger's review of "" the watcher "" ( un...",0,negative,neg/cv076_26009.txt
4,neg_cv417_14653,the marvelous british actor derek jacobi stars...,0,negative,neg/cv417_14653.txt


In [7]:
X_hidden = vectorizer.transform(hidden_test_df["text"])
hidden_preds = clf.predict(X_hidden)

hidden_preds[:10]


array([0, 0, 1, 1, 1, 1, 1, 1, 1, 1])

In [8]:
hidden_acc = accuracy_score(hidden_test_df["label"], hidden_preds)
hidden_cm = confusion_matrix(hidden_test_df["label"], hidden_preds)
hidden_cm_df = pd.DataFrame(
    hidden_cm,
    index=["true negative", "true positive"],
    columns=["pred negative", "pred positive"],
)

print(f"Hidden test accuracy: {hidden_acc:.4f}")
print()
print(classification_report(hidden_test_df["label"], hidden_preds, target_names=["negative", "positive"]))
print("Confusion matrix:")
print(hidden_cm_df)

Hidden test accuracy: 0.6833

              precision    recall  f1-score   support

    negative       0.83      0.46      0.59       300
    positive       0.63      0.90      0.74       300

    accuracy                           0.68       600
   macro avg       0.73      0.68      0.67       600
weighted avg       0.73      0.68      0.67       600

Confusion matrix:
               pred negative  pred positive
true negative            139            161
true positive             29            271


In [9]:
public_test_acc = stage1_config["public_test_accuracy"]

comparison_df = pd.DataFrame({
    "dataset": ["public_test.csv (Stage 1)", "hidden_test.csv (Stage 2)"],
    "n_examples": [400, len(hidden_test_df)],
    "accuracy": [public_test_acc, hidden_acc],
})
print(comparison_df.to_string(index=False))
print()
print(f"Difference (hidden - public): {hidden_acc - public_test_acc:+.4f}")

                  dataset  n_examples  accuracy
public_test.csv (Stage 1)         400  0.715000
hidden_test.csv (Stage 2)         600  0.683333

Difference (hidden - public): -0.0317


In [10]:
predictions_df = pd.DataFrame({
    "id": hidden_test_df["id"],
    "predicted_label": hidden_preds,
})
predictions_df["predicted_label"] = predictions_df["predicted_label"].astype(int)
predictions_df.to_csv("hidden_test_predictions.csv", index=False)
predictions_df.head()


,id,predicted_label
0,neg_cv795_10291,0
1,neg_cv174_9735,0
2,pos_cv065_15248,1
3,neg_cv076_26009,1
4,neg_cv417_14653,1


In [11]:
if IN_COLAB:
    from google.colab import files
    files.download("hidden_test_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>